In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 第6周 Day 5：Agent 开发实战与框架设计

> **导语**：前四天我们学了 Agent 的概念、工具调用、ReAct 模式和推理评测。今天进入"工程师时间"——怎么从零搭建一个生产级 Agent 系统？框架怎么选？状态怎么管？错误怎么处理？日志怎么记？这篇文章把"玩具 Agent"变成"生产 Agent"。

---


## 📊 学习进度


In [ ]:
███████████████████████████████████░░░░░░░░░░░░░░░░░ 54%
第1周 ✅  Python基础
第2周 ✅  AI与大模型基础
第3周 ✅  RAG检索增强生成
第4周 ✅  向量数据库与Embedding
第5周 ✅  大模型微调与部署
第6周 🔄  Agent与工具使用（Day 5/7）← 今天


---


## 一、为什么需要 Agent 框架？

### 1.1 "玩具" vs "生产"的差距

你跟着教程写了一个 50 行的 Agent，能查天气、能算数，跑通了！然后部署到生产环境，马上被现实教做人：

- **LLM 超时**：OpenAI API 突然不响应了，用户等了 30 秒看到报错
- **工具崩溃**：天气 API 挂了，Agent 直接卡死
- **参数错误**：LLM 提取的参数格式不对，工具执行报异常
- **上下文溢出**：多轮对话后，上下文超出了模型窗口限制
- **无法调试**：出问题了不知道哪一步错的，日志一片空白
- **并发问题**：多个用户同时访问，状态互相串了

**生产级 Agent 框架**就是来解决这些问题的。它不是一个"Agent"，而是一个"Agent 运行环境"。

### 1.2 Agent 框架的核心职责

| 职责 | 说明 | 类比 |
|------|------|------|
| **流程控制** | 管理 ReAct 循环、终止条件 | 像操作系统调度进程 |
| **状态管理** | 管理对话历史、任务状态 | 像数据库管理数据 |
| **错误处理** | 重试、降级、兜底 | 像保险机制 |
| **可观测性** | 日志、追踪、监控 | 像仪表盘 |
| **并发管理** | 多用户、多会话隔离 | 像容器隔离 |
| **安全控制** | 权限、审计、过滤 | 像门禁系统 |

### 1.3 生活类比——"餐厅管理系统"

一个 Agent 框架就像一个**餐厅管理系统**：
- **LLM = 主厨**：决策做什么菜、怎么做
- **工具 = 厨房设备**：烤箱、炒锅、冰箱（各有用途）
- **执行引擎 = 出餐流程**：点单→备料→烹饪→质检→出餐
- **状态管理 = 订单本**：记住每桌点了什么、做到哪一步
- **错误处理 = Plan B**：烤箱坏了就改用炒锅，食材不够就换菜单
- **监控 = 餐厅大屏**：实时看到运营状况

---


## 二、核心原理详解

### 2.1 Agent 框架核心三要素

**要素一：LLM 大脑（决策层）**

负责理解意图、选择工具、生成回复。框架需要：
- 支持**多模型切换**（GPT/Qwen/GLM/DeepSeek）
- 处理 **token 限制**（超长上下文截断或摘要）
- 管理 **温度参数**（推理任务低温、创意任务高温）
- 支持 **流式输出**（边生成边返回，提升用户体验）

**要素二：工具库（能力层）**

负责与外部世界交互。框架需要：
- **工具注册**：标准化方式定义和注册工具
- **参数验证**：执行前校验参数类型和格式
- **执行隔离**：一个工具出错不影响其他工具
- **动态加载**：根据场景加载不同工具集

**要素三：执行引擎（控制层）**

负责编排 Agent 的执行流程。框架需要：
- **Agent Loop**：实现 ReAct 循环（思考→行动→观察→重复）
- **状态机**：管理任务的不同状态（待处理→执行中→已完成→失败）
- **超时控制**：每一步都有超时限制，防卡死
- **并发控制**：异步执行、并行工具调用

### 2.2 状态管理策略

Agent 需要记住很多东西：对话历史、用户信息、任务进度、中间结果。不同的存储策略适用于不同阶段：

| 策略 | 存储 | 速度 | 持久性 | 适用场景 |
|------|------|------|--------|---------|
| **内存** | Python dict | ⚡⚡⚡ | ❌ 重启丢失 | 开发调试、单次会话 |
| **文件** | JSON/SQLite | ⚡⚡ | ✅ 本地持久化 | 小型应用、单机部署 |
| **数据库** | PostgreSQL/Redis | ⚡ | ✅ 高可靠 | 生产环境、多用户 |
| **向量库** | ChromaDB/Milvus | ⚡ | ✅ 长期记忆 | RAG增强、经验积累 |

**核心原则**：
- **短期记忆**（当前对话）→ 用 LLM 上下文窗口
- **中期记忆**（用户偏好）→ 用 Redis（快速读写）
- **长期记忆**（历史摘要）→ 用 PostgreSQL + 向量库

**记忆管理策略——"漏斗模式"**：


In [ ]:
全量对话历史（短期）
       ↓ 压缩/摘要
关键信息提取（中期）
       ↓ 向量化存储
知识沉淀（长期）


### 2.3 错误处理与重试

生产环境中，工具调用一定会失败。必须处理四种典型错误：

| 错误类型 | 原因 | 处理策略 |
|---------|------|---------|
| **网络超时** | API 响应慢或不可达 | 指数退避重试（1s→2s→4s） |
| **参数错误** | LLM 提取的参数不合法 | 反馈给 LLM 让它修正，不自动重试 |
| **服务不可用** | 工具所在服务宕机 | 降级到备选工具或人工兜底 |
| **结果异常** | 返回了意外格式的数据 | 数据校验 + 格式适配 |

**指数退避重试（Exponential Backoff）**：


In [ ]:
第1次尝试 → 失败 → 等 1 秒
第2次尝试 → 失败 → 等 2 秒
第3次尝试 → 失败 → 等 4 秒
第4次尝试 → 放弃，走降级方案


为什么用指数退避而不是固定等待？因为如果是服务过载导致的失败，固定间隔重试只会让服务更忙。指数退避给服务"喘息"的时间。

### 2.4 可观测性设计

生产级 Agent 必须能"看见"自己在做什么。可观测性三支柱：

**日志（Logging）**：记录每一步操作


In [ ]:
# 好的日志格式
2025-07-21 10:30:15 [INFO] Agent received user message: "查天气"
2025-07-21 10:30:15 [INFO] LLM decided to use tool: get_weather
2025-07-21 10:30:15 [INFO] Tool get_weather called with args: {"city": "北京"}
2025-07-21 10:30:16 [INFO] Tool get_weather returned: {"temp": 32, "condition": "晴"}
2025-07-21 10:30:16 [INFO] Response generated in 1.2s


**追踪（Tracing）**：可视化整个请求链路


In [ ]:
用户请求 → Agent接收 → LLM决策(0.8s) → 工具调用(0.3s) → 结果整合(0.5s) → 返回
     总耗时: 1.6s | 各环节耗时一目了然


**指标（Metrics）**：统计数据
- 平均响应时间
- 工具调用成功率
- 每日活跃用户数
- Token 消耗量

---


## 三、主流框架对比与选择

### 3.1 五大 Agent 框架横评

| 框架 | 优点 | 缺点 | 最适合 |
|------|------|------|--------|
| **LangChain + LangGraph** | 生态最丰富、社区活跃、集成多 | 抽象层厚、调试困难 | 快速原型、多工具集成 |
| **OpenAI Agents SDK** | 官方支持、简洁直观 | 绑定OpenAI生态 | GPT为主的简单Agent |
| **AutoGen (Microsoft)** | 多Agent协作优秀 | 学习曲线陡峭 | 多Agent对话/协作 |
| **CrewAI** | 角色定义直观 | 功能相对简单 | 角色扮演式多Agent |
| **自研轻量框架** | 完全可控、按需定制 | 开发成本高 | 特定业务、高定制需求 |

### 3.2 选型决策树


In [ ]:
你的需求是什么？
├── 快速验证想法（MVP）
│   └── → LangChain（生态全、示例多）
├── 简单的单Agent客服
│   └── → OpenAI Agents SDK（简洁够用）
├── 多Agent协作系统
│   ├── 对话式协作 → AutoGen
│   └── 角色分工 → CrewAI
├── 企业级生产系统
│   └── → 自研框架 + 开源组件（完全可控）
└── 不确定？
    └── → 先用 LangChain 快速验证，再决定是否自研


---


## 四、代码实战

### 4.1 自研轻量 Agent 框架核心


In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt
import numpy as np
import time
import json
from datetime import datetime
from typing import Dict, List, Any, Optional, Callable
from collections import defaultdict

# matplotlib 中文字体配置
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False


# ===== 模块一：带重试的工具执行器 =====
class RobustToolExecutor:
    """带指数退避重试和降级机制的工具执行器"""
    
    def __init__(self, max_retries=3, base_delay=1.0):
        self.max_retries = max_retries     # 最大重试次数
        self.base_delay = base_delay       # 初始延迟
        self.call_log = []                 # 调用日志（可观测性）
    
    def execute(self, tool_name: str, args: dict, 
                tool_fn: Callable, fallback_fn: Callable = None) -> dict:
        """
        执行工具调用，自动重试
        - tool_fn: 主工具函数
        - fallback_fn: 降级方案（主函数失败后执行）
        """
        last_error = None
        
        for attempt in range(1, self.max_retries + 1):
            try:
                start_time = time.time()
                result = tool_fn(**args)      # 执行工具
                elapsed = time.time() - start_time
                
                # 记录成功日志
                log_entry = {
                    "tool": tool_name,
                    "args": args,
                    "attempt": attempt,
                    "status": "success",
                    "elapsed": f"{elapsed:.3f}s",
                    "timestamp": datetime.now().isoformat()
                }
                self.call_log.append(log_entry)
                
                return {"status": "success", "data": result, "attempts": attempt}
                
            except TimeoutError as e:
                last_error = e
                print(f"  ⚠️ {tool_name} 超时（第{attempt}次）")
            except Exception as e:
                last_error = e
                error_msg = str(e)
                
                # 参数错误不重试！反馈给 LLM 修正
                if "invalid" in error_msg.lower() or "type" in error_msg.lower():
                    print(f"  ❌ {tool_name} 参数错误，不重试")
                    log_entry = {
                        "tool": tool_name, "args": args,
                        "attempt": attempt, "status": "param_error",
                        "error": error_msg,
                        "timestamp": datetime.now().isoformat()
                    }
                    self.call_log.append(log_entry)
                    return {"status": "param_error", "error": error_msg}
                
                print(f"  ⚠️ {tool_name} 执行失败（第{attempt}次）：{error_msg}")
            
            # 指数退避等待（最后一次不用等）
            if attempt < self.max_retries:
                delay = self.base_delay * (2 ** (attempt - 1))
                print(f"  ⏳ 等待 {delay}s 后重试...")
                time.sleep(delay)
        
        # 所有重试都失败，尝试降级方案
        if fallback_fn:
            print(f"  🔄 所有重试失败，执行降级方案...")
            try:
                result = fallback_fn(**args)
                self.call_log.append({
                    "tool": tool_name, "args": args,
                    "status": "fallback", "timestamp": datetime.now().isoformat()
                })
                return {"status": "fallback", "data": result}
            except:
                pass
        
        return {"status": "failed", "error": str(last_error)}


# ===== 模块二：Agent 调用追踪器 =====
class AgentTracer:
    """Agent 执行追踪器：记录每一步操作，用于调试和监控"""
    
    def __init__(self):
        self.traces = []
        self.current_trace = None
    
    def start_trace(self, trace_id: str, user_message: str):
        """开始追踪一次请求"""
        self.current_trace = {
            "trace_id": trace_id,
            "user_message": user_message,
            "start_time": datetime.now().isoformat(),
            "spans": [],
            "status": "running"
        }
    
    def add_span(self, name: str, span_type: str, data: Any = None):
        """添加一个追踪片段"""
        if self.current_trace:
            self.current_trace["spans"].append({
                "name": name,
                "type": span_type,
                "data": data,
                "timestamp": datetime.now().isoformat()
            })
    
    def end_trace(self, status: str = "success"):
        """结束追踪"""
        if self.current_trace:
            self.current_trace["status"] = status
            self.current_trace["end_time"] = datetime.now().isoformat()
            self.traces.append(self.current_trace)
            self.current_trace = None
    
    def print_report(self):
        """打印追踪报告"""
        for trace in self.traces[-3:]:  # 最近3条
            print(f"\n📋 Trace {trace['trace_id']}: {trace['user_message'][:30]}...")
            print(f"   状态: {trace['status']}")
            for span in trace["spans"]:
                icon = {"decision": "🧠", "tool_call": "🔧", 
                       "response": "💬", "error": "❌"}.get(span["type"], "•")
                print(f"   {icon} {span['name']}")
            start = datetime.fromisoformat(trace["start_time"])
            end = datetime.fromisoformat(trace["end_time"])
            duration = (end - start).total_seconds()
            print(f"   ⏱️ 总耗时: {duration:.2f}s")


# ===== 模块三：Agent 核心 =====
class ProductionAgent:
    """生产级 Agent（简化版）"""
    
    def __init__(self):
        self.executor = RobustToolExecutor(max_retries=3, base_delay=0.1)
        self.tracer = AgentTracer()
        self.conversation_history = []
        self.trace_counter = 0
    
    def chat(self, user_message: str) -> str:
        """处理用户消息"""
        self.trace_counter += 1
        trace_id = f"trace-{self.trace_counter:04d}"
        
        self.tracer.start_trace(trace_id, user_message)
        self.tracer.add_span("收到用户消息", "decision", user_message)
        
        # 记录对话历史（状态管理）
        self.conversation_history.append({"role": "user", "content": user_message})
        
        # 模拟工具调用（实际由 LLM 决策）
        def mock_tool(**kwargs):
            return f"查询结果: {kwargs}"
        
        result = self.executor.execute(
            "query_tool",
            {"query": user_message[:20]},
            mock_tool
        )
        
        self.tracer.add_span(f"工具调用: query_tool", "tool_call", result)
        
        # 生成回复
        response = f"已处理您的请求（{result['status']}）"
        self.conversation_history.append({"role": "assistant", "content": response})
        self.tracer.add_span("生成回复", "response", response)
        
        self.tracer.end_trace()
        return response


# 运行框架
print("🏗️ 生产级 Agent 框架演示\n")
agent = ProductionAgent()

agent.chat("北京天气怎么样？")
agent.chat("帮我算一下 2+3*4")
agent.chat("查一下库存")

print("\n" + "=" * 50)
print("📊 追踪报告：")
agent.tracer.print_report()


### 4.2 状态管理策略对比可视化


In [ ]:
# 四种状态管理策略对比图
fig, ax = plt.subplots(figsize=(14, 7))

strategies = ['内存\n(dict/list)', '文件\n(JSON/SQLite)', '数据库\n(PostgreSQL/Redis)', '向量库\n(ChromaDB)']
dimensions = {
    '读写速度': [10, 7, 5, 4],
    '数据容量': [2, 5, 9, 10],
    '持久性': [1, 7, 9, 8],
    '开发复杂度\n(越低越好)': [1, 4, 8, 7],
}

x = np.arange(len(strategies))
width = 0.18
colors = ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350']

for i, (dim, values) in enumerate(dimensions.items()):
    bars = ax.bar(x + i * width, values, width, label=dim, color=colors[i], alpha=0.85)
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height + 0.2,
                str(int(height)), ha='center', va='bottom', fontsize=9, fontweight='bold')

# 标注推荐场景
ax.annotate('← 开发调试', xy=(0, 11), fontsize=10, color='#1565C0', fontweight='bold', ha='center')
ax.annotate('← 生产环境', xy=(2, 11), fontsize=10, color='#2E7D32', fontweight='bold', ha='center')

ax.set_xlabel('状态管理策略', fontsize=13)
ax.set_ylabel('评分（1-10）', fontsize=13)
ax.set_title('Agent 状态管理策略全面对比', fontsize=15, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(strategies, fontsize=11)
ax.legend(fontsize=11, loc='upper right')
ax.set_ylim(0, 12)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/root/learning-notebooks/第6周/ima/day5_state_management.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 状态管理对比图已保存")


---


## 五、业务关联

### 5.1 LangChat 平台架构与框架设计

LangChat 作为企业 AI 平台，其 Agent 系统采用了分层架构：


In [ ]:
┌─────────────────────────────────┐
│       用户接口层                 │  ← Web/API/微信/钉钉
├─────────────────────────────────┤
│       Agent 编排层               │  ← 任务拆分、Agent调度
├─────────────────────────────────┤
│       工具系统层                 │  ← 工具注册、参数验证、执行引擎
├─────────────────────────────────┤
│       LLM 接入层                 │  ← 多模型支持、负载均衡
├─────────────────────────────────┤
│       数据与状态层               │  ← 对话历史、用户画像、知识库
├─────────────────────────────────┤
│       基础设施层                 │  ← 日志、监控、部署、安全
└─────────────────────────────────┘


### 5.2 糖水店 Agent 框架设计

如果给 Jason 的糖水店设计一个完整的 Agent 系统：

**工具集设计**：
- `query_order`：查询订单状态
- `check_inventory`：检查原料库存
- `create_order`：创建新订单
- `process_payment`：处理支付
- `send_notification`：发送通知（短信/微信）
- `generate_report`：生成销售报表
- `search_knowledge`：搜索糖水店知识库

**状态管理设计**：
- 短期记忆：当前对话上下文（内存，3轮对话）
- 中期记忆：用户偏好和订单历史（Redis，TTL 24小时）
- 长期记忆：客户画像和消费记录（PostgreSQL + 向量库）

**错误处理设计**：
- 支付失败 → 自动重试3次 → 降级到人工处理
- 库存查询超时 → 返回缓存数据 + 提示"数据可能有延迟"
- LLM 决策异常 → 走预设的安全回复

---


## 六、常见误区

### ❌ 误区一："用 LangChain 就行了，不用自研"
**真相**：LangChain 非常适合快速原型验证。但到了生产环境，它的抽象层太多，调试困难，版本更新频繁（经常 breaking change）。很多团队最终会从 LangChain 迁移到自研轻量框架。建议：MVP 用 LangChain，生产考虑自研或 LangGraph。

### ❌ 误区二："Agent 不需要状态管理，上下文窗口够大"
**真相**：即使 128K 的上下文窗口也装不下多轮对话+工具结果+知识检索。而且上下文越长，响应越慢、成本越高。状态管理不是"够不够"的问题，是"性能和成本优化"的问题。

### ❌ 误区三："错误处理就是 try-except"
**真相**：好的错误处理包括：分类（网络错误/参数错误/业务错误）、策略（重试/降级/兜底）、反馈（告诉 LLM 出了什么错让它调整）、记录（日志和追踪）。一个 `try-except` 只是最基础的。

### ❌ 误区四："先把功能做完，日志以后再加"
**真相**：没有日志的 Agent 在生产环境中是"黑盒"——出问题了完全不知道为什么。可观测性应该在架构设计阶段就考虑好，不是后补的。每个工具调用、每次 LLM 决策都要有日志。

---


## 七、课堂练习（5分钟）

**练习一**：框架选择题

你要做一个多 Agent 协作的销售助手，每个 Agent 负责不同产品线（糖水/饮品/甜品），需要互相通信。推荐用哪个框架？为什么？

**练习二**：设计状态管理方案

设计一个"智能客服 Agent"的状态管理方案，要求：
- 能记住当前对话（多轮）
- 能记住用户的投诉历史（跨会话）
- 能检索相关产品知识（RAG）

请回答：每类记忆用什么存储？为什么？

**练习三**：思考题

为什么"参数错误"不适合自动重试，而"网络超时"适合？

---


## 八、课后测试（10分钟）

**1. Agent 框架的核心三要素是什么？**
A. LLM + 提示词 + 温度参数
B. LLM 大脑 + 工具库 + 执行引擎
C. 输入层 + 处理层 + 输出层
D. 前端 + 后端 + 数据库

**2. 以下哪种存储最适合"短期对话记忆"？**
A. PostgreSQL
B. Python dict（内存）
C. 向量数据库
D. 硬盘文件

**3. 指数退避重试的等待时间序列是？**
A. 1s, 1s, 1s, 1s（固定）
B. 1s, 2s, 4s, 8s（指数增长）
C. 4s, 3s, 2s, 1s（递减）
D. 随机时间

**4. 可观测性的三支柱是什么？**
A. 日志 + 追踪 + 指标
B. 代码 + 测试 + 文档
C. 输入 + 处理 + 输出
D. 速度 + 质量 + 成本

**5. 参数错误不适合自动重试的原因是？**
A. 浪费时间
B. 同样的输入会产生同样的错误参数
C. 会损坏系统
D. 用户会等太久

---


## 九、术语表

| 英文术语 | 音标 | 中文释义 |
|----------|------|----------|
| Framework | /ˈfreɪmwɜːk/ | 框架，提供基础设施的开发工具集 |
| State Management | /steɪt ˈmænɪdʒmənt/ | 状态管理，维护Agent的上下文和历史 |
| Exponential Backoff | /ˌekspəˈnenʃəl ˈbækɒf/ | 指数退避，重试间隔逐渐增加 |
| Fallback | /ˈfɔːlbæk/ | 降级方案，主方案失败后的备选 |
| Observability | /əbˌzɜːvəˈbɪləti/ | 可观测性，了解系统内部状态的能力 |
| Tracing | /ˈtreɪsɪŋ/ | 追踪，跟踪请求在系统中的流转 |
| Orchestration | /ˌɔːkɪˈstreɪʃən/ | 编排，协调多个组件协作执行 |
| Resilience | /rɪˈzɪliəns/ | 弹性/容错性，系统在故障下继续运行 |
| Idempotent | /ˌaɪdemˈpəʊtənt/ | 幂等性，多次执行结果相同 |
| Graceful Degradation | /ˈɡreɪsfəl ˌdeɡrəˈdeɪʃən/ | 优雅降级，出错时仍有基本功能 |

---


## 十、参考资源

### 📹 视频推荐
1. **《Building Effective AI Agents》— LangChain 团队**（约45分钟）
   推荐搜索：building effective AI agents langchain

2. **《从零搭建 AI Agent》系列教程**
   推荐搜索：AI agent from scratch tutorial

### 📖 延伸阅读
1. **LangGraph 官方文档**（新一代 Agent 编排框架）
   https://langchain-ai.github.io/langgraph/

2. **Awesome AI Agents — Agent 生态汇总**
   https://github.com/e2b-dev/awesome-ai-agents

---

> 📅 **明天预告**：Day 6 是**纯代码实战日**——我们要把今天学的框架设计落地，从零搭建一个完整的 Function Calling Agent！工具定义、执行层、Agent 核心、多轮对话测试，全程代码实操，不写虚的！
